# Musanga V4 — NB04 | Independent paired RGB A/B evaluation (v4_001)

**Question:** Does adding exhaustive plot labels to the same baseline circle + partial crown supervision improve spatially independent RGB U-Net predictions?

**Frozen:** NB01 50-m buffered, 3-fold split; NB03 checkpoints/cache; 30 circles and 76 RGB-covered plots (Mixt1/Mixt2 excluded). **No retraining.** Predict both arms on the **same raster pixels** of each held-out unit. Circle truth: KVO exhaustive crowns + background within complete circle. Plot truth: Nadine `Class_ID==1` + background within the corresponding plot **only under NB03's provisional exhaustiveness assumption**. Explicit class negatives and geometry conflicts are masked. Do not call training partial annotations ground truth outside exhaustive footprints.

Outputs: unit metrics, paired A–B deltas, site/type summaries, cover strata, scatterplots, probability histograms, provenance and resumable per-unit artifacts. Memory-bounded windowed raster inference, no whole mosaic loading.

## 00 — Environment, saved training contract and frozen geometry

In [ ]:
%pip -q install geopandas pyogrio shapely rasterio scipy pandas scikit-learn matplotlib segmentation-models-pytorch
from pathlib import Path
import json, hashlib, gc, warnings, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import transform as shape_transform
from pyproj import Transformer
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from scipy.special import expit
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')] if p.exists()),None)
assert ROOT is not None, 'Musanga Final folder not found'
OLD=ROOT/'Outputs/V3_1'; AUDIT=ROOT/'Outputs/v4/00_input_audit'
SPLIT_ROOT=ROOT/'Outputs/v4/01_spatial_cover_folds_v2'
NB02=ROOT/'Outputs/v4/02_rgb_circles_plus_plots_v3'
NB03=ROOT/'Outputs/v4/03_paired_rgb_unet_v1'
OUT=ROOT/'Outputs/v4/04_paired_rgb_heldout_v1';OUT.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
SPLIT_CONTRACT=json.loads((SPLIT_ROOT/'01_reference_fold_contract_v2.json').read_text())
TRAIN=json.loads((NB03/'03_training_contract.json').read_text())
FOLD_CSV=Path(SPLIT_CONTRACT['output_csv'])
assert hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()==SPLIT_CONTRACT['csv_sha256']
assert TRAIN['frozen_nb01_hash']==SPLIT_CONTRACT['csv_sha256']
SPLITS=pd.read_csv(FOLD_CSV)
assert len(SPLITS)==108 and (SPLITS.unit_type=='circle').sum()==30 and (SPLITS.unit_type=='plot').sum()==78
UNITS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
assert len(UNITS)==108 and not UNITS.duplicated(['unit_type','unit_id']).any()
OWNERS=pd.read_csv(NB02/'02_reference_mosaic_owners_v2.csv')
assert not OWNERS.duplicated(['unit_type','unit_id']).any()
EXCLUDED={('plot','Mixt1'),('plot','Mixt2')}
assert set(map(tuple,OWNERS.loc[OWNERS.roi_mosaic_coverage_fraction.lt(.95),['unit_type','unit_id']].astype(str).values))==EXCLUDED
EVAL=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
    on=['unit_type','unit_id'],validate='one_to_one')
EVAL=EVAL[EVAL.roi_mosaic_coverage_fraction.ge(.95)].copy()
assert len(EVAL)==106 and (EVAL.unit_type=='circle').sum()==30 and (EVAL.unit_type=='plot').sum()==76
assert set(TRAIN['checkpoints'])=={str(NB03/f'fold{f}_{a}_{TRAIN["training_key"][:18]}.pt') for f in range(3) for a in 'AB'}
assert all(Path(p).is_file() for p in TRAIN['checkpoints'])
MOSAICS={}
for q in V31['mosaics']:
    if q['file'].endswith('.tif'):
        p=Path(q['path'])
        if not p.exists(): p=ROOT/'Data/Ortho_RGB'/q['file']
        assert p.exists(),p
        MOSAICS[q['file']]=p
assert len(MOSAICS)==6
print('HELD-OUT UNITS:',EVAL.groupby(['fold','site','unit_type']).size().to_string())
print('CHECKPOINTS:',len(TRAIN['checkpoints']),'TRAINING KEY:',TRAIN['training_key'][:18]);print('OUTPUT:',OUT)

## 01 — Geometry provenance; mask contradictory labels

In [ ]:
from shapely import make_valid
from shapely.ops import unary_union
from shapely.geometry import GeometryCollection
REF_CRS='EPSG:32635'
def repair(g):
    if g is None or g.is_empty: raise ValueError('Empty reference geometry')
    if not g.is_valid:
        z=make_valid(g)
        pieces=[h for h in getattr(z,'geoms',[z]) if h.geom_type in ('Polygon','MultiPolygon')]
        g=unary_union(pieces)
    assert g.is_valid and not g.is_empty
    return g
def union(g):
    arr=list(g)
    return unary_union(arr) if arr else GeometryCollection()
CIRCLE_PATH=Path(F78['circle_complete_roi']); EXH_PATH=Path(F78['circle_positive_crowns'])
PLOT_PATH=Path(F78['plot_file']); CROWN_PATH=Path(F78['exhaustive_plot_crowns'])
circle_src=gpd.read_file(CIRCLE_PATH).to_crs(REF_CRS);circle_src['unit_id']=circle_src.index.astype(str)
plot_src=gpd.read_file(PLOT_PATH).to_crs(REF_CRS);plot_src['unit_id']=plot_src.Plot_name.astype(str)
assert len(circle_src)==30 and len(plot_src)==78
# Repair using identical NB03 polygonal semantics, with explicit identifiers.
circle_src.geometry=[repair(g) for g in circle_src.geometry]
plot_src.geometry=[repair(g) for g in plot_src.geometry]
exh=gpd.read_file(EXH_PATH).to_crs(REF_CRS);exh.geometry=[repair(g) for g in exh.geometry]
crowns=gpd.read_file(CROWN_PATH).to_crs(REF_CRS);crowns.geometry=[repair(g) for g in crowns.geometry]
crowns['is_musanga']=pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)
KVO_POS=union(exh.geometry)
NADINE_POS=union(crowns.loc[crowns.is_musanga].geometry)
NADINE_NEG=union(crowns.loc[~crowns.is_musanga].geometry)
ROI=gpd.read_file(Path(V31['roi'])).to_crs(REF_CRS)
ROI_GEOM=union(ROI.geometry)
# Unions are never used to redefine unit identity. Each unit retains its own extent.
REF_UNITS={('circle',str(z.unit_id)):z.geometry for z in circle_src.itertuples()}
REF_UNITS.update({('plot',str(z.unit_id)):z.geometry for z in plot_src.itertuples()})
assert all((str(z.unit_type),str(z.unit_id)) in REF_UNITS for z in EVAL.itertuples())
# Exclude only genuine annotation disagreement on circle/plot shared footprints.
SHARED=union(circle_src.geometry).intersection(union(plot_src.geometry))
CONFLICT=(KVO_POS.intersection(SHARED).symmetric_difference(
    NADINE_POS.intersection(SHARED))).union(NADINE_POS.intersection(NADINE_NEG))
assert CONFLICT.is_valid
print('REFERENCE:',len(REF_UNITS),'| conflict area m²:',round(CONFLICT.area,1))
print('WARNING: plot background = absent Musanga only if annotations are exhaustive, as assumed by NB03.')

## 02 — Windowed RGB inference; same pixels for both models

In [ ]:
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from scipy.ndimage import binary_dilation
TILE=256;STRIDE=128;BATCH=8
PROB_THRESHOLD=.5
# NB03 uses RGB.astype(float32)/255; keep exactly that input scaling.
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
    def forward(self,x): return self.net(x)[:,0]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
def get_models(fold):
    models={}
    for arm in 'AB':
        path=NB03/f'fold{fold}_{arm}_{TRAIN["training_key"][:18]}.pt'
        ck=torch.load(path,map_location='cpu',weights_only=False)
        assert ck['fold']==fold and ck['arm']==arm and ck['training_key']==TRAIN['training_key']
        assert int(ck['epoch'])==12
        m=ResNet18UNet()
        m.load_state_dict(ck['model'],strict=True)
        m=m.to(DEVICE).eval();models[arm]=m
    return models

def to_ds_geom(g,ds):
    if ds.crs==rasterio.crs.CRS.from_user_input(REF_CRS):return g
    tx=Transformer.from_crs(REF_CRS,ds.crs,always_xy=True)
    return shape_transform(tx.transform,g)
def burn(g,ds,win):
    shape=(int(win.height),int(win.width))
    if g is None or g.is_empty:return np.zeros(shape,bool)
    bbox=box(*rasterio.windows.bounds(win,ds.transform))
    if not g.intersects(bbox):return np.zeros(shape,bool)
    clipped=g.intersection(bbox)
    return rasterize([(clipped,1)],out_shape=shape,
                     transform=window_transform(win,ds.transform),fill=0,dtype='uint8').astype(bool)
def rgb_read(ds,win):
    data=ds.read([1,2,3],window=win,masked=True)
    valid=~np.ma.getmaskarray(data).any(axis=0)
    valid &= np.isfinite(np.asarray(data.data,dtype=np.float32)).all(axis=0)
    if ds.count>=4:
        alpha=ds.read(4,window=win,masked=True)
        valid &= ~np.ma.getmaskarray(alpha) & (np.asarray(alpha.filled(0))>0)
    x=np.clip(np.asarray(data.filled(0)),0,255).astype('uint8').transpose(1,2,0)
    return x,valid

def origins(length,tile=TILE,stride=STRIDE):
    if length<=tile:return [0]
    vals=list(range(0,length-tile+1,stride))
    if vals[-1]!=length-tile:vals.append(length-tile)
    return vals
# Positive Hann with non-zero boundary weights; invariant to A/B.
w=np.hanning(TILE+2)[1:-1]
HANN=np.outer(w,w).astype('float32');HANN/=HANN.max()
@torch.inference_mode()
def infer_one_unit(ds,unit_geom,models):
    geom=to_ds_geom(unit_geom,ds)
    bounds=rasterio.windows.from_bounds(*geom.bounds,transform=ds.transform)
    # Integral bounding box, 1 tile halo. Clip to mosaic edges; pad every tile if needed.
    row0=max(0,int(np.floor(bounds.row_off))-TILE)
    col0=max(0,int(np.floor(bounds.col_off))-TILE)
    row1=min(ds.height,int(np.ceil(bounds.row_off+bounds.height))+TILE)
    col1=min(ds.width,int(np.ceil(bounds.col_off+bounds.width))+TILE)
    h=row1-row0;ww=col1-col0
    if h<=0 or ww<=0:raise RuntimeError('No image pixels in unit bbox')
    if h*ww>75_000_000:raise RuntimeError('Unit bbox unexpectedly huge; inspect geometry')
    sums={a:np.zeros((h,ww),np.float32) for a in 'AB'}
    weights=np.zeros((h,ww),np.float32)
    # Cover all of the target bbox with 256x256 tiles.
    windows=[];patches=[]
    def flush():
        if not patches:return
        inputs=torch.from_numpy(np.stack(patches).astype('float32')/255.).to(DEVICE)
        for arm,model in models.items():
            with torch.autocast('cuda',enabled=DEVICE=='cuda'):
                probs=torch.sigmoid(model(inputs)).float().cpu().numpy()
            for (r,c,hh,wwi),p in zip(windows,probs):
                sums[arm][r:r+hh,c:c+wwi]+=p[:hh,:wwi]*HANN[:hh,:wwi]
        for r,c,hh,wwi in windows:
            weights[r:r+hh,c:c+wwi]+=HANN[:hh,:wwi]
        patches.clear();windows.clear()
    for r in origins(h):
        for c in origins(ww):
            hh=min(TILE,h-r);wwi=min(TILE,ww-c)
            tile,valid=rgb_read(ds,Window(col0+c,row0+r,wwi,hh))
            full=np.zeros((TILE,TILE,3),np.uint8)
            full[:hh,:wwi]=tile
            # NB03 always trained on in-bounds 256px chips; exclude invalid model output later.
            patches.append(full.transpose(2,0,1));windows.append((r,c,hh,wwi))
            if len(patches)==BATCH:flush()
    flush()
    assert np.all(weights>0),'Coverage gap in tile blending'
    final={arm:np.clip(sums[arm]/weights,0,1).astype('float32') for arm in 'AB'}
    win=Window(col0,row0,ww,h)
    _,valid=rgb_read(ds,win)
    unit=burn(geom,ds,win)
    roi=burn(to_ds_geom(ROI_GEOM,ds),ds,win)
    mask=valid&unit&roi
    # Full-model input everywhere, common evaluation support; no partial reference truth.
    return final,mask,win
print('DEVICE:',DEVICE,'tile',TILE,'stride',STRIDE,'batch',BATCH)

## 03 — Held-out unit truth and paired metrics

In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score
HIST_BINS=np.linspace(0,1,101)
def truth_mask(unit_type,ds,win,unit_mask):
    # Positives come only from the selected exhaustive reference source.
    positive=KVO_POS if unit_type=='circle' else NADINE_POS
    pos=burn(to_ds_geom(positive,ds),ds,win)&unit_mask
    conflict=burn(to_ds_geom(CONFLICT,ds),ds,win)
    supervised=unit_mask&~conflict
    # No background labels inferred from non-exhaustive circles.
    if unit_type=='plot':
        # Explicit negative crowns are contradictory with positive crowns when overlapping.
        positive_negative=burn(to_ds_geom(NADINE_NEG,ds),ds,win)&pos
        supervised&=~positive_negative
    return pos,supervised

def scores(y,p):
    y=np.asarray(y,dtype=bool);p=np.asarray(p,dtype=np.float32)
    q=p>=PROB_THRESHOLD;tp=int((q&y).sum());fp=int((q&~y).sum());fn=int((~q&y).sum());tn=int((~q&~y).sum())
    f1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan
    iou=tp/(tp+fp+fn) if (tp+fp+fn) else np.nan
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    obs=100*y.mean();pred=100*p.mean()
    out=dict(n_pixels=len(y),n_pos=int(y.sum()),n_neg=int((~y).sum()),
             observed_cover_pp=obs,predicted_cover_pp=pred,cover_error_pp=pred-obs,
             abs_cover_error_pp=abs(pred-obs),brier=float(np.mean((p-y)**2)),
             tp=tp,fp=fp,fn=fn,tn=tn,F1=f1,IoU=iou,precision=precision,recall=recall)
    if y.any() and (~y).any():
        out['roc_auc']=float(roc_auc_score(y,p));out['average_precision']=float(average_precision_score(y,p))
    else:out.update(roc_auc=np.nan,average_precision=np.nan)
    hist_pos=np.histogram(p[y],bins=HIST_BINS)[0]
    hist_neg=np.histogram(p[~y],bins=HIST_BINS)[0]
    return out,hist_pos,hist_neg

PER_UNIT=OUT/'per_unit';PER_UNIT.mkdir(exist_ok=True)
SCHEMA='nb04_paired_rgb_heldout_v1'
def unit_path(z):
    return PER_UNIT/f'fold{int(z.fold)}_{z.unit_type}_{str(z.unit_id).replace("/","_")}.npz'
def infer_and_save(z,models):
    f=unit_path(z)
    if f.exists():
        with np.load(f,allow_pickle=False) as obj:
            if str(obj['training_key'].item())==TRAIN['training_key'] and str(obj['schema'].item())==SCHEMA:
                return 'cached'
        raise ValueError(f'Provenance mismatch in {f}; refuse silent overwrite')
    with rasterio.open(MOSAICS[z.mosaic]) as ds:
        predictions,common,win=infer_one_unit(ds,REF_UNITS[(str(z.unit_type),str(z.unit_id))],models)
        true,mask=truth_mask(str(z.unit_type),ds,win,common)
        if mask.sum()==0:raise RuntimeError(f'Zero supervised evaluation pixels: {z.unit_type} {z.unit_id}')
        y=true[mask].astype('uint8')
        a=predictions['A'][mask].astype('float32')
        b=predictions['B'][mask].astype('float32')
    # Histograms + sufficiently accurate sum for unit cover and Brier; save sampled scatter inputs separately.
    rA,hpA,hnA=scores(y,a);rB,hpB,hnB=scores(y,b)
    meta={k:str(getattr(z,k)) for k in ['unit_type','unit_id','site','mosaic','spatial_group']}
    meta['fold']=int(z.fold)
    payload=dict(schema=np.array(SCHEMA),training_key=np.array(TRAIN['training_key']),
        metadata=np.array(json.dumps(meta)),A=np.array(json.dumps(rA)),B=np.array(json.dumps(rB)),
        A_positive_hist=hpA,A_negative_hist=hnA,B_positive_hist=hpB,B_negative_hist=hnB)
    # Atomic writes are resume-safe; never destroy an already verified result.
    import os
    tmp=f.with_name(f.name+'.partial.npz');np.savez_compressed(tmp,**payload);os.replace(tmp,f)
    return 'new'

FORCE_RECOMPUTE=False  # If changing inference/masks, create a NEW output directory rather than overwriting.
assert not FORCE_RECOMPUTE
results=[]
for fold in range(3):
    print('FOLD',fold,'loading paired checkpoints')
    models=get_models(fold)
    for z in EVAL[EVAL.fold.eq(fold)].itertuples(index=False):
        status=infer_and_save(z,models)
        results.append(dict(fold=fold,site=z.site,unit_type=z.unit_type,unit_id=str(z.unit_id),status=status))
        print(f'  {z.site:9s} {z.unit_type:6s} {str(z.unit_id):20s} {status}',flush=True)
    del models;gc.collect()
    if DEVICE=='cuda':torch.cuda.empty_cache()
print('FINISHED:',pd.DataFrame(results).groupby(['fold','status']).size().to_string())
assert len(results)==106

## 04 — Paired results, stratification, and bootstrap confidence intervals

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
rows=[];hist=[]
for z in EVAL.itertuples(index=False):
    p=unit_path(z)
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as a:
        assert str(a['training_key'].item())==TRAIN['training_key']
        meta=json.loads(str(a['metadata'].item()))
        for arm in 'AB':
            d=json.loads(str(a[arm].item()))
            rows.append(dict(**meta,arm=arm,**d))
            hist.append(dict(**meta,arm=arm,pos=a[f'{arm}_positive_hist'].copy(),neg=a[f'{arm}_negative_hist'].copy()))
M=pd.DataFrame(rows)
assert len(M)==212 and M.groupby(['fold','unit_type','unit_id']).arm.nunique().eq(2).all()
M.to_csv(OUT/'04_heldout_unit_metrics.csv',index=False)
keys=['fold','site','unit_type','unit_id','spatial_group']
P=M.pivot(index=keys,columns='arm')
flat=P.copy();flat.columns=[f'{k}_{a}' for k,a in flat.columns]
flat=flat.reset_index()
for col in ['F1','IoU','precision','recall','brier','roc_auc','average_precision','abs_cover_error_pp','cover_error_pp','predicted_cover_pp']:
    flat[f'delta_B_minus_A_{col}']=flat[f'{col}_B']-flat[f'{col}_A']
flat.to_csv(OUT/'04_paired_unit_differences.csv',index=False)

def unit_summary(frame):
    vals=[]
    for arm,sub in frame.groupby('arm'):
        # Macro-average unit F1, not site-size-weighted pixel pooling.
        e=sub.predicted_cover_pp-sub.observed_cover_pp
        vals.append(dict(arm=arm,n_units=len(sub),mean_F1=sub.F1.mean(),mean_IoU=sub.IoU.mean(),
          mean_precision=sub.precision.mean(),mean_recall=sub.recall.mean(),
          cover_MAE_pp=float(np.abs(e).mean()),cover_RMSE_pp=float(np.sqrt((e**2).mean())),
          cover_bias_pp=float(e.mean()),brier=sub.brier.mean()))
    return pd.DataFrame(vals)
summary=[]
for (site,typ),sub in M.groupby(['site','unit_type']):
    x=unit_summary(sub);x['site']=site;x['unit_type']=typ;summary.append(x)
summary=pd.concat(summary,ignore_index=True)
summary.to_csv(OUT/'04_site_type_summary.csv',index=False)
print('SITE × REFERENCE TYPE:')
display(summary[['site','unit_type','arm','n_units','mean_F1','mean_IoU','cover_MAE_pp','cover_RMSE_pp','cover_bias_pp']].round(3))
print('ALL CIRCLES versus ALL PLOTS (do not mix source definitions):')
for typ in ['circle','plot']:
    print(typ);display(unit_summary(M[M.unit_type.eq(typ)]).round(3))
# Whole-unit bootstrap respects pairing; report intervals by reference type.
rng=np.random.default_rng(20261118)
boot=[]
for typ,sub in flat.groupby('unit_type'):
    for metric in ['F1','IoU','abs_cover_error_pp','brier']:
        v=sub[f'delta_B_minus_A_{metric}'].dropna().to_numpy()
        if len(v)<2:continue
        sims=np.mean(v[rng.integers(len(v),size=(3000,len(v)))],axis=1)
        boot.append(dict(unit_type=typ,metric=metric,n=len(v),delta_B_minus_A=v.mean(),
                         ci_low=np.percentile(sims,2.5),ci_high=np.percentile(sims,97.5)))
boot=pd.DataFrame(boot);boot.to_csv(OUT/'04_paired_bootstrap_CI.csv',index=False)
display(boot.round(4))

## 05 — Cover classes, scatterplots and calibration figures

In [ ]:
# Strata use REFERENCE cover only; identical assignments for A and B.
strat=flat.copy();strat['reference_cover_pp']=strat['observed_cover_pp_A']
assert np.allclose(strat['observed_cover_pp_A'],strat['observed_cover_pp_B'])
strat['cover_stratum']=pd.cut(strat.reference_cover_pp,bins=[-0.001,1,5,20,40,100.001],
                             labels=['0–1%','1–5%','5–20%','20–40%','>40%'],include_lowest=True)
long=M.merge(strat[keys+['cover_stratum']],on=keys,validate='many_to_one')
strat_summary=long.groupby(['unit_type','cover_stratum','arm'],observed=True).agg(
    n=('unit_id','size'),F1=('F1','mean'),IoU=('IoU','mean'),
    cover_MAE_pp=('abs_cover_error_pp','mean'),cover_bias_pp=('cover_error_pp','mean')).reset_index()
strat_summary.to_csv(OUT/'04_cover_stratum_summary.csv',index=False)
display(strat_summary.round(3))
fig,axs=plt.subplots(2,2,figsize=(12,11),constrained_layout=True)
for i,typ in enumerate(['circle','plot']):
    df=M[M.unit_type.eq(typ)]
    for j,arm in enumerate('AB'):
        ax=axs[i,j];z=df[df.arm.eq(arm)]
        for site,q in z.groupby('site'):
            ax.scatter(q.observed_cover_pp,q.predicted_cover_pp,label=f'{site} (n={len(q)})',alpha=.8,s=36)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),xlabel='Reference Musanga cover (%)',
               ylabel='Predicted Musanga cover (%)',title=f'{typ.capitalize()} held-out — model {arm}')
        ax.grid(alpha=.15);ax.legend(fontsize=8)
fig.savefig(OUT/'04_cover_scatter_by_reference_and_arm.png',dpi=180)
plt.show()
fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,typ in zip(axs,['circle','plot']):
    q=flat[flat.unit_type.eq(typ)]
    for site,s in q.groupby('site'):
        ax.scatter(s['abs_cover_error_pp_A'],s['abs_cover_error_pp_B'],label=site,s=35,alpha=.8)
    ax.plot([0,100],[0,100],'k--',lw=1)
    ax.set(xlim=(0,60),ylim=(0,60),xlabel='Model A absolute cover error (pp)',
           ylabel='Model B absolute cover error (pp)',title=f'{typ.capitalize()} (below line = B better)')
    ax.grid(alpha=.15);ax.legend(fontsize=8)
fig.savefig(OUT/'04_paired_absolute_cover_error.png',dpi=180)
plt.show()

## 06 — Aggregated ROC/PR histograms and final quality gates

In [ ]:
from sklearn.metrics import roc_curve,precision_recall_curve,auc
# Pool per-unit probability histograms WITHOUT rereading imagery. Approximate curves at 0.01 bins.
centres=(HIST_BINS[:-1]+HIST_BINS[1:])/2
fig,axs=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
for i,typ in enumerate(['circle','plot']):
    for arm in 'AB':
        q=[v for v in hist if v['unit_type']==typ and v['arm']==arm]
        pos=np.sum([x['pos'] for x in q],axis=0).astype(float)
        neg=np.sum([x['neg'] for x in q],axis=0).astype(float)
        tp=np.r_[pos.sum(),np.cumsum(pos[::-1])[:-1]][::-1] if False else np.r_[0,np.cumsum(pos[::-1])]
        fp=np.r_[0,np.cumsum(neg[::-1])]
        tpr=tp/max(pos.sum(),1); fpr=fp/max(neg.sum(),1)
        precision=tp/np.maximum(tp+fp,1)
        recall=tpr
        axs[i,0].plot(fpr,tpr,label=f'{arm}, AUC≈{auc(fpr,tpr):.3f}')
        axs[i,1].plot(recall,precision,label=arm)
    axs[i,0].plot([0,1],[0,1],'k--',lw=.6)
    axs[i,0].set(title=f'{typ.capitalize()} pooled ROC (binned)',xlabel='FPR',ylabel='TPR')
    axs[i,1].set(title=f'{typ.capitalize()} pooled PR (binned)',xlabel='Recall',ylabel='Precision')
    for ax in axs[i]:ax.legend();ax.grid(alpha=.15)
fig.savefig(OUT/'04_roc_pr_binned.png',dpi=180);plt.show()
report=dict(schema=SCHEMA,nb01_hash=SPLIT_CONTRACT['csv_sha256'],
            nb03_training_key=TRAIN['training_key'],n_units=len(flat),n_circles=30,n_plots=76,
            excluded=['Mixt1','Mixt2'],pixel_threshold=PROB_THRESHOLD,tile=TILE,stride=STRIDE,
            notes=['Same evaluation pixels for A and B','Plot background exhaustiveness is provisional',
                   'Unannotated partial crowns are not evaluation negatives',
                   'Held-out 50 m spatial buffers inherited from NB01/NB03'],
            metric_csv=str(OUT/'04_heldout_unit_metrics.csv'))
(OUT/'04_evaluation_contract.json').write_text(json.dumps(report,indent=2))
assert len(M)==212 and M.groupby(['unit_type','unit_id']).arm.nunique().eq(2).all()
print('NB04 COMPLETE:',OUT)
print('CHECK: 30 circles + 76 plots, A/B matched, 3 spatial folds. No checkpoint updated.')